In [2]:
!pip install boto3 botocore

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.9/14.9 MB 20.0 MB/s eta 0:00:00a 0:00:01

[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [ ]:
import os
import boto3
import botocore
from botocore.client import BaseClient

# ─────────────────────────────────────────────────────────────
# Buckets
# ─────────────────────────────────────────────────────────────
MY_STORAGE   = "my-storage"
GROUP1_BUCKET = "group1-bucket"
GROUP2_BUCKET = "group2-bucket"


# ─────────────────────────────────────────────────────────────
# Helpers
# ─────────────────────────────────────────────────────────────
def make_client(
    access_key: str | None = None,
    secret_key: str | None = None,
    endpoint: str | None = None,
    region: str | None = None,
) -> BaseClient:
    """
    Create an S3 client using either explicit keys or the current environment.
    """
    access_key = access_key or os.environ["AWS_ACCESS_KEY_ID"]
    secret_key = secret_key or os.environ["AWS_SECRET_ACCESS_KEY"]
    endpoint   = endpoint   or os.environ["AWS_S3_ENDPOINT"]
    region     = region     or os.environ["AWS_DEFAULT_REGION"]

    session = boto3.Session(
        aws_access_key_id=access_key,
        aws_secret_access_key=secret_key,
        region_name=region,
    )
    return session.client(
        "s3",
        endpoint_url=endpoint,
        config=botocore.client.Config(signature_version="s3v4"),
    )


def run(label: str, expect_success: bool, fn, *args, **kwargs):
    """Run one S3 call and print a normalised OK / FAILED line."""
    outcome = "SUCCESS" if expect_success else "FAIL"
    print(f"  [{label}] expect {outcome:<7} ", end="")
    try:
        result = fn(*args, **kwargs)
        if expect_success:
            print("→ OK")
        else:
            print("→ !! UNEXPECTED SUCCESS — check your policies")
        return result
    except botocore.exceptions.ClientError as e:
        code = e.response["Error"]["Code"]
        if not expect_success:
            print(f"→ OK (denied: {code})")
        else:
            print(f"→ FAILED ({code}: {e.response['Error'].get('Message', '')})")
    except Exception as e:
        if not expect_success:
            print(f"→ OK (denied: {type(e).__name__})")
        else:
            print(f"→ FAILED ({type(e).__name__}: {e})")


# ─────────────────────────────────────────────────────────────
# Per-user test suites
# ─────────────────────────────────────────────────────────────
def test_admin(s3: BaseClient):
    """
    admin — full access to every bucket, no restrictions.
    """
    print("\n╔══ ADMIN — full access to all buckets ══╗")

    for bucket in (MY_STORAGE, GROUP1_BUCKET, GROUP2_BUCKET):
        print(f"\n  ── {bucket} ──")
        run(f"list   {bucket}", True,  s3.list_objects_v2, Bucket=bucket, MaxKeys=5)
        run(f"put    {bucket}", True,  s3.put_object,      Bucket=bucket, Key="probe.txt", Body=b"test")
        run(f"delete {bucket}", True,  s3.delete_object,   Bucket=bucket, Key="probe.txt")


def test_user1(s3: BaseClient):
    """
    user1 — ReadOnly on my-storage, FullAccess on teama-bucket, NoAccess on teamb-bucket.
    """
    print("\n╔══ USER1 — readonly:my-storage | full:teama-bucket | none:teamb-bucket ══╗")

    # my-storage: list → allow, put → deny
    print(f"\n  ── {MY_STORAGE} (readonly) ──")
    run(f"list   {MY_STORAGE}", True,  s3.list_objects_v2, Bucket=MY_STORAGE, MaxKeys=5)
    run(f"put    {MY_STORAGE}", False, s3.put_object,      Bucket=MY_STORAGE, Key="probe.txt", Body=b"test")

    # teama-bucket: full access
    print(f"\n  ── {GROUP1_BUCKET} (full access) ──")
    run(f"list   {GROUP1_BUCKET}", True,  s3.list_objects_v2, Bucket=GROUP1_BUCKET, MaxKeys=5)
    run(f"put    {GROUP1_BUCKET}", True,  s3.put_object,      Bucket=GROUP1_BUCKET, Key="probe.txt", Body=b"test")
    run(f"delete {GROUP1_BUCKET}", True,  s3.delete_object,   Bucket=GROUP1_BUCKET, Key="probe.txt")

    # teamb-bucket: no access
    print(f"\n  ── {GROUP2_BUCKET} (no access) ──")
    run(f"list   {GROUP2_BUCKET}", False, s3.list_objects_v2, Bucket=GROUP2_BUCKET, MaxKeys=5)
    run(f"put    {GROUP2_BUCKET}", False, s3.put_object,      Bucket=GROUP2_BUCKET, Key="probe.txt", Body=b"test")


def test_user2(s3: BaseClient):
    """
    user2 — ReadOnly on my-storage, NoAccess on teama-bucket, FullAccess on teamb-bucket.
    """
    print("\n╔══ USER2 — readonly:my-storage | none:teama-bucket | full:teamb-bucket ══╗")

    # my-storage: list → allow, put → deny
    print(f"\n  ── {MY_STORAGE} (readonly) ──")
    run(f"list   {MY_STORAGE}", True,  s3.list_objects_v2, Bucket=MY_STORAGE, MaxKeys=5)
    run(f"put    {MY_STORAGE}", False, s3.put_object,      Bucket=MY_STORAGE, Key="probe.txt", Body=b"test")

    # teama-bucket: no access
    print(f"\n  ── {GROUP1_BUCKET} (no access) ──")
    run(f"list   {GROUP1_BUCKET}", False, s3.list_objects_v2, Bucket=GROUP1_BUCKET, MaxKeys=5)
    run(f"put    {GROUP1_BUCKET}", False, s3.put_object,      Bucket=GROUP1_BUCKET, Key="probe.txt", Body=b"test")

    # teamb-bucket: full access
    print(f"\n  ── {GROUP2_BUCKET} (full access) ──")
    run(f"list   {GROUP2_BUCKET}", True,  s3.list_objects_v2, Bucket=GROUP2_BUCKET, MaxKeys=5)
    run(f"put    {GROUP2_BUCKET}", True,  s3.put_object,      Bucket=GROUP2_BUCKET, Key="probe.txt", Body=b"test")
    run(f"delete {GROUP2_BUCKET}", True,  s3.delete_object,   Bucket=GROUP2_BUCKET, Key="probe.txt")


# ─────────────────────────────────────────────────────────────
# Entry point
# ─────────────────────────────────────────────────────────────
if __name__ == "__main__":
    # Use current env creds; you can run separately as admin / user1 / user2.
    s3_client = make_client()

    # Choose which suite to run based on an env var, e.g. TEST_USER=admin|user1|user2
    which = os.environ.get("TEST_USER", "admin").lower()
    # which = os.environ.get("TEST_USER", "user1").lower()
    # which = os.environ.get("TEST_USER", "user2").lower()

    if which == "admin":
        test_admin(s3_client)
    elif which == "user1":
        test_user1(s3_client)
    elif which == "user2":
        test_user2(s3_client)
    else:
        raise ValueError(f"Unknown TEST_USER value: {which}")


╔══ USER2 — readonly:my-storage | none:teama-bucket | full:teamb-bucket ══╗

  ── my-storage (readonly) ──
  [list   my-storage] expect SUCCESS → OK
  [put    my-storage] expect FAIL    → OK (denied: AccessDenied)

  ── group1-bucket (no access) ──
  [list   group1-bucket] expect FAIL    → OK (denied: AccessDenied)
  [put    group1-bucket] expect FAIL    → OK (denied: AccessDenied)

  ── group2-bucket (full access) ──
  [list   group2-bucket] expect SUCCESS → OK
  [put    group2-bucket] expect SUCCESS → OK
  [delete group2-bucket] expect SUCCESS → OK
